In [2]:
from google_play_scraper import app, Sort, reviews  # <-- Use reviews(), not reviews_all
import pandas as pd

banks = {
    "CBE": "com.combanketh.mobilebanking",
    "BOA": "com.boa.boaMobileBanking",
    "Dashen": "com.dashen.dashensuperapp"
}

all_reviews = []

for bank_name, app_id in banks.items():
    print(f"Fetching reviews for: {bank_name} ({app_id})")
    
    # Fetch exactly 400 reviews per bank
    reviews_batch, continuation_token = reviews(
        app_id,
        lang='en',
        country='et',  
        count=500,      # <-- Limits to 500 reviews
        sort=Sort.NEWEST,
    )
    
    for review in reviews_batch:
        all_reviews.append({
            "bank": bank_name,
            "review": review["content"],
            "rating": review["score"],
            "date": review["at"].strftime("%Y-%m-%d"),
            "source": "Google Play"
        })

# Save to CSV
df = pd.DataFrame(all_reviews)
df.to_csv("bank_reviews_limited.csv", index=False)
print(f"Total reviews collected: {len(df)} (Expected: 1,200 = 400 x 3)")
print(df["bank"].value_counts())  # Verify counts per bank

Fetching reviews for: CBE (com.combanketh.mobilebanking)
Fetching reviews for: BOA (com.boa.boaMobileBanking)
Fetching reviews for: Dashen (com.dashen.dashensuperapp)
Total reviews collected: 1454 (Expected: 1,200 = 400 x 3)
bank
CBE       500
BOA       500
Dashen    454
Name: count, dtype: int64


In [3]:
import pandas as pd

# Load the scraped data
df = pd.read_csv("bank_reviews_limited.csv")

# 1. Remove duplicates
df = df.drop_duplicates(subset=["review", "bank"])

# 2. Handle missing values
df = df.dropna(subset=["review"])  # Drop empty reviews
df["rating"] = df["rating"].fillna(0)  # Fill missing ratings with 0

# 3. Save cleaned data
df.to_csv("cleaned_bank_reviews.csv", index=False)
print(f"Cleaned reviews: {len(df)}")

Cleaned reviews: 1234


In [4]:
# Check total reviews
assert len(df) >= 1200, f"Only {len(df)} reviews scraped (needed: 1200+)."

# Check missing data (<5%)
missing_percentage = df.isnull().mean().max() * 100
assert missing_percentage < 5, f"Missing data: {missing_percentage:.2f}% (max allowed: 5%)."

In [5]:
# Convert the date column to datetime (pandas handles most common formats automatically)
df['normalized_date'] = pd.to_datetime(df['date'], errors='coerce')

# Format as 'YYYY-MM-DD' (ISO standard)
df['normalized_date'] = df['normalized_date'].dt.strftime('%Y-%m-%d')
# 4. Overwrite the original file
df.to_csv('cleaned_bank_reviews.csv', index=False)
print("Original file updated with normalized dates!")

Original file updated with normalized dates!
